In [ ]:
from google.colab import userdata

!pip install psycopg2-binary

import psycopg2
import sys
from IPython.display import display, clear_output
import ipywidgets as widgets

#Menu
class MenuPrzyciskow:


    def __init__(self, BazaDnaych):

        self.baza = BazaDnaych

        self.pole_imie= widgets.Text(description="Imię:")
        self.pole_nazwisko = widgets.Text(description="Nazwisko:")
        self.pole_telefon = widgets.Text(description="Telefon:")
        self.przycisk_zapisz = widgets.Button(description="Dodaj klienta", button_style="success", icon="check")

        self.IDdoUsunięcia= widgets.Text(description="ID do usunięcia:")
        self.przycisk_usun = widgets.Button(description="Usuń klienta", button_style="danger", icon="trash")

        self.wyjscie_dodawania = widgets.Output()



        self.przycisk_zapisz.on_click(self.ZapiszKlienta)
        widok_dodawania = widgets.VBox([
            self.pole_imie,
            self.pole_nazwisko,
            self.pole_telefon,
            self.przycisk_zapisz,
            self.wyjscie_dodawania
        ])


        self.przycisk_odswiez = widgets.Button(description="Odśwież listę", button_style="info", icon="refresh")
        self.wyjscie_listy = widgets.Output()

        self.przycisk_odswiez.on_click(self.OdswiezListy)

        widok_listy = widgets.VBox([
            self.przycisk_odswiez,
            self.wyjscie_listy
        ])

        self.przycisk_usun.on_click(self.UsuwanieZListyPoID)
        self.wyjscie_usuwania = widgets.Output()

        widok_UsuwanieKlienta = widgets.VBox([
            self.IDdoUsunięcia,
            self.przycisk_usun,
            self.wyjscie_usuwania
        ])

        self.przycisk_Szukaj = widgets.Button(description="Szukaj klienta", button_style="info", icon="search")
        self.Opcja_Wyszukiwania= widgets.Dropdown(options=["Imieniu", "Nazwisku", "ID"], description="Wybierz kryterium wyszukiwania:")
        self.Wyszukiwarka= widgets.Text(description="Wyszukaj klienta:")
        self.wyjscie_szukaj = widgets.Output()

        self.przycisk_Szukaj.on_click(self.WyszukajKlienta)

        widok_WyszukiwarkaKlientow = widgets.VBox([

            self.Opcja_Wyszukiwania,
            self.Wyszukiwarka,
            self.przycisk_Szukaj,
            self.wyjscie_szukaj
        ])
        self.zakladki = widgets.Tab()

        self.zakladki.children = [widok_dodawania, widok_listy, widok_UsuwanieKlienta, widok_WyszukiwarkaKlientow]

        self.zakladki.set_title(0, "Dodaj Klienta")
        self.zakladki.set_title(1, "Lista Klientów")
        self.zakladki.set_title(2, "Usuń Klienta")
        self.zakladki.set_title(3, "Wyszukaj Klienta")

        self.OdswiezListy(None)


    def PokazMenu(self):
       display(self.zakladki)


    def ZapiszKlienta(self, przycisk):

        imie = self.pole_imie.value
        nazwisko = self.pole_nazwisko.value
        telefon = self.pole_telefon.value


        with self.wyjscie_dodawania:
            clear_output()

            if imie and nazwisko:
                self.baza.DodajKlienta(imie, nazwisko, telefon)
                print(f"Dodano klienta {imie} {nazwisko}")


                self.pole_imie.value = ""
                self.pole_nazwisko.value = ""
                self.pole_telefon.value = ""
                self.OdswiezListy(None)
            else:
                print(" Błąd: Imię i Nazwisko są wymagane.")


    def OdswiezListy(self, przycisk):

        with self.wyjscie_listy:
            clear_output()
            print("Aktualna lista klientów w bazie danych:\n")
            self.baza.PokazListeKlientow()

        with self.wyjscie_usuwania:
            clear_output()
            print("Aktualna lista klientów w bazie danych:\n")
            self.baza.PokazListeKlientow()


    def UsuwanieZListyPoID(self, przycisk):
      id_do_usuniecia = self.IDdoUsunięcia.value
      with self.wyjscie_usuwania:
        clear_output()
        if id_do_usuniecia:
            try:
                self.baza.KlientUsun(id_do_usuniecia)
                print(f" Sukces: Usunięto klienta o ID: {id_do_usuniecia}")
                self.IDdoUsunięcia.value = ""
                self.OdswiezListy(None)

            except Exception as e:
                print(f"Błąd: {e}")
                self.baza.PokazListeKlientow()
        else:
            print(" Błąd: Musisz podać ID klienta, aby go usunąć.")


    def WyszukajKlienta(self, przycisk):
        wybor = self.Opcja_Wyszukiwania.value
        WpisWyszukiwania = self.Wyszukiwarka.value
        with self.wyjscie_szukaj:
          clear_output()

          if not WpisWyszukiwania:
            print("Musisz podać coś do wyszukiwania")
            return

          print(f"Wynik wyszukiwania dla {WpisWyszukiwania}")
          Wynik = self.baza.WyszukajKlientaMechanizm(wybor, WpisWyszukiwania)

          if not Wynik:
            print("Brak Klienta")
            return
          print(f"Znaleziono klientów:")
          print("")
          for client in Wynik:
                  print(f"ID: {client[0]} | Imię i Nazwisko: {client[1]} {client[2]} | Tel: {client[3]}")
          print("-----------------------")







class SystemZapisow:

    def __init__(self):

        #Łączenie z bazą
        url_bazy = userdata.get("DATABASE_URL")

        try:
            self.conn = psycopg2.connect(url_bazy)
            self.cursor = self.conn.cursor()
            print("Połączono z Bazą")

            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS klienci (
                    id SERIAL PRIMARY KEY,
                    imie TEXT,
                    nazwisko TEXT,
                    telefon TEXT)
            """)
            self.conn.commit()

        except Exception as e:

            print(f"\nBłąd połączenia z bazą: {e}")
            print("Zatrzymuję program.")
            sys.exit()

    def DodajKlienta(self, imie, nazwisko, telefon):
        try:
          self.cursor.execute("INSERT INTO klienci (imie, nazwisko, telefon) VALUES (%s, %s, %s)", (imie, nazwisko, telefon))
          self.conn.commit()
          print("Dodano klienta do bazy.")
        except Exception as e:
          self.conn.rollback()
          print(f"Błąd bazy danych: {e}")

    def PokazListeKlientow(self):
        try:
          self.cursor.execute("SELECT * FROM klienci")
          clients = self.cursor.fetchall()

          if not clients:

              print("Baza danych jest pusta.")
              return

          for client in clients:

              print(f"ID: {client[0]} | Imię i Nazwisko: {client[1]} {client[2]} | Tel: {client[3]}")
          print("-----------------------")
        except Exception as e:
          print(f"Błąd bazy danych: {e}")


    def KlientUsun(self, id_klienta):
      try:
        self.cursor.execute("DELETE FROM klienci WHERE id = %s", (int(id_klienta),))
        self.conn.commit()
      except ValueError:
        raise ValueError("ID klienta musi być liczbą całkowitą.")
      except Exception as e:
        print(f"Błąd bazy danych: {e}")

    def WyszukajKlientaMechanizm(self, tryb, WpisWyszukiwania):
      try:
        if tryb == "Imieniu":
          self.cursor.execute("SELECT * FROM klienci WHERE imie ILIKE %s", (f"%{WpisWyszukiwania}%",))


        elif tryb == "Nazwisku":
          self.cursor.execute("SELECT * FROM klienci WHERE nazwisko ILIKE %s", (f"%{WpisWyszukiwania}%",))


        elif tryb == "ID":
          try:
            self.cursor.execute("SELECT * FROM klienci WHERE id = %s", (WpisWyszukiwania,))
          except ValueError:
                  return []

        return self.cursor.fetchall()
      except Exception as e:
        self.conn.rollback()
        print(f"Błąd bazy danych")
        return []






BazaDanych = SystemZapisow()
Start=MenuPrzyciskow(BazaDanych)
Start.PokazMenu()